In [1]:
import pandas as pd
import geopandas as gpd


In [5]:
total_occurrences = pd.read_csv("/run/media/vincent/Extreme Pro/MyBiodiversity/Biodiversity/Resubmit/Optimized/data/total_presence_points.csv")
total_occurrences_gdf = gpd.GeoDataFrame(total_occurrences, geometry=gpd.points_from_xy(total_occurrences.longitude, total_occurrences.latitude))
total_occurrences_gdf.head()

,longitude,latitude,geometry
0,37.237427,-12.385247,POINT (37.23743 -12.38525)
1,35.006272,-15.783571,POINT (35.00627 -15.78357)
2,35.304866,-15.364770,POINT (35.30487 -15.36477)
3,37.037940,0.122650,POINT (37.03794 0.12265)
4,35.200451,-18.044269,POINT (35.20045 -18.04427)


In [ ]:
CSV_PATH = "/run/media/vincent/Extreme Pro/MyBiodiversity/Biodiversity/Resubmit/Optimized/data/total_presence_points.csv"

data = pd.read_csv(CSV_PATH)

data_gdf = gpd.GeoDataFrame(data, geometry=gpd.points_from_xy(data.longitude, data.latitude))

kenya_area_of_interest = gpd.read_file("/run/media/vincent/Extreme Pro/MyBiodiversity/Biodiversity/Resubmit/Optimized/data/kenya_area_of_interest.shp")

In [5]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pystac_client import Client
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

CSV_PATH = "/run/media/vincent/Extreme Pro/MyBiodiversity/Biodiversity/Resubmit/Optimized/data/Kenya_total_presence_points.csv"

OUTPUT_CSV = "/run/media/vincent/Extreme Pro/MyBiodiversity/Biodiversity/Resubmit/Optimized/data/Kenya_occurrences_with_geomad_2024.csv"

YEAR = "2024"

CHUNK_SIZE = 500

os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

# ============================================================
# LOAD POINTS
# ============================================================

print("Loading points...")

df = pd.read_csv(CSV_PATH)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df.longitude,
        df.latitude
    ),
    crs="EPSG:4326"
)

print(f"Loaded {len(gdf):,} points")

xmin, ymin, xmax, ymax = gdf.total_bounds

print("\nBounds:")
print(xmin, ymin, xmax, ymax)

# ============================================================
# CONNECT TO STAC
# ============================================================

print("\nConnecting to DEA STAC...")

catalog = Client.open(
    "https://explorer.digitalearth.africa/stac"
)

# ============================================================
# SEARCH WITH RETRIES
# ============================================================

print("\nSearching GeoMAD tiles...")

items = None

for attempt in range(5):

    try:

        search = catalog.search(
            collections=["gm_s2_annual"],
            bbox=[xmin, ymin, xmax, ymax],
            datetime=f"{YEAR}-01-01/{YEAR}-12-31",
            limit=100
        )

        items = list(search.items())

        print(
            f"Found {len(items)} GeoMAD tiles"
        )

        break

    except Exception as e:

        print(
            f"Attempt {attempt + 1}/5 failed:"
        )
        print(e)

        time.sleep(10)

if items is None:
    raise RuntimeError(
        "Unable to retrieve GeoMAD tiles."
    )

# ============================================================
# BANDS
# ============================================================

bands = [
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B11",
    "B12",
    "SMAD",
    "EMAD",
    "BCMAD"
]

# ============================================================
# BUILD BAND URL INDEX
# ============================================================

print("\nBuilding tile index...")

tile_assets = {
    band: []
    for band in bands
}

for item in items:

    for band in bands:

        if band not in item.assets:
            continue

        url = item.assets[band].href

        if url.startswith(
            "s3://deafrica-services/"
        ):
            url = url.replace(
                "s3://deafrica-services/",
                "https://deafrica-services.s3.af-south-1.amazonaws.com/"
            )

        tile_assets[band].append(url)

for band in bands:
    print(
        f"{band}: {len(tile_assets[band])} tiles"
    )

# ============================================================
# POINT COORDINATES
# ============================================================

coords = list(
    zip(
        gdf.geometry.x.values,
        gdf.geometry.y.values
    )
)

# ============================================================
# EXTRACTION FUNCTION
# ============================================================

def sample_band_urls(urls, coords):

    out = np.full(
        len(coords),
        np.nan,
        dtype="float32"
    )

    remaining = np.arange(
        len(coords)
    )

    for url in urls:

        if len(remaining) == 0:
            break

        try:

            with rasterio.open(url) as src:

                pts = [
                    coords[i]
                    for i in remaining
                ]

                vals = np.array(
                    [
                        x[0]
                        for x in src.sample(pts)
                    ],
                    dtype="float32"
                )

                nodata = src.nodata

                if nodata is not None:
                    valid = vals != nodata
                else:
                    valid = np.isfinite(vals)

                idx = remaining[valid]

                out[idx] = vals[valid]

                remaining = remaining[
                    ~valid
                ]

        except Exception as e:

            print(
                f"Failed tile:\n{url}"
            )

            continue

    return out

# ============================================================
# EXTRACT VALUES
# ============================================================

results = pd.DataFrame(
    index=df.index
)

start = time.time()

for band in bands:

    print(
        f"\n{'='*60}\n"
        f"Processing {band}\n"
        f"{'='*60}"
    )

    urls = tile_assets[band]

    if len(urls) == 0:

        print(
            f"No assets found for {band}"
        )

        results[band] = np.nan

        continue

    values = np.full(
        len(coords),
        np.nan,
        dtype="float32"
    )

    for i in tqdm(
        range(
            0,
            len(coords),
            CHUNK_SIZE
        )
    ):

        chunk = coords[
            i:i + CHUNK_SIZE
        ]

        vals = sample_band_urls(
            urls,
            chunk
        )

        values[
            i:i + len(chunk)
        ] = vals

    results[band] = values

    valid_count = np.isfinite(
        values
    ).sum()

    elapsed = (
        time.time() - start
    ) / 60

    print(
        f"Valid values: "
        f"{valid_count:,}/{len(values):,}"
    )

    print(
        f"Elapsed: "
        f"{elapsed:.1f} minutes"
    )

# ============================================================
# SAVE
# ============================================================

print("\nSaving output...")

final_df = pd.concat(
    [df, results],
    axis=1
)

final_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nFinished successfully")
print(f"Saved to:\n{OUTPUT_CSV}")

print("\nMissing values summary:")
print(
    final_df[bands]
    .isna()
    .sum()
    .sort_values()
)

Loading points...
Loaded 623 points

Bounds:
34.02929549 -4.58009211 40.93005124 4.284146667

Connecting to DEA STAC...

Searching GeoMAD tiles...
Found 104 GeoMAD tiles

Building tile index...
B02: 104 tiles
B03: 104 tiles
B04: 104 tiles
B05: 104 tiles
B06: 104 tiles
B07: 104 tiles
B08: 104 tiles
B8A: 104 tiles
B11: 104 tiles
B12: 104 tiles
SMAD: 104 tiles
EMAD: 104 tiles
BCMAD: 104 tiles

Processing B02


100%|██████████| 2/2 [02:13<00:00, 66.65s/it] 


Valid values: 0/623
Elapsed: 2.2 minutes

Processing B03


100%|██████████| 2/2 [01:51<00:00, 55.84s/it] 


Valid values: 0/623
Elapsed: 4.1 minutes

Processing B04


...

Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x220/y071/2024--P1Y/gm_s2_annual_x220y071_2024--P1Y_EMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x216/y082/2024--P1Y/gm_s2_annual_x216y082_2024--P1Y_EMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x218/y076/2024--P1Y/gm_s2_annual_x218y076_2024--P1Y_EMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x216/y078/2024--P1Y/gm_s2_annual_x216y078_2024--P1Y_EMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x217/y077/2024--P1Y/gm_s2_annual_x217y077_2024--P1Y_EMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x215/y074/2024--P1Y/gm_s2_annual_x215y074_2024--P1Y_EMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x216/y079/2024--P1Y/gm_s2_annual

  0%|          | 0/2 [00:00<?, ?it/s]

Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x218/y073/2024--P1Y/gm_s2_annual_x218y073_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x217/y078/2024--P1Y/gm_s2_annual_x217y078_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x219/y070/2024--P1Y/gm_s2_annual_x219y070_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x215/y072/2024--P1Y/gm_s2_annual_x215y072_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x216/y075/2024--P1Y/gm_s2_annual_x216y075_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x221/y079/2024--P1Y/gm_s2_annual_x221y079_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x220/y081/2024--P1Y/gm_s2_

100%|██████████| 2/2 [00:00<00:00, 33.31it/s]

Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x215/y071/2024--P1Y/gm_s2_annual_x215y071_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x219/y071/2024--P1Y/gm_s2_annual_x219y071_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x216/y080/2024--P1Y/gm_s2_annual_x216y080_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x221/y075/2024--P1Y/gm_s2_annual_x221y075_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x221/y081/2024--P1Y/gm_s2_annual_x221y081_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x215/y082/2024--P1Y/gm_s2_annual_x215y082_2024--P1Y_BCMAD.tif
Failed tile:
https://deafrica-services.s3.af-south-1.amazonaws.com/gm_s2_annual/1-0-0/x220/y080/2024--P1Y/gm_s2_